# Train YOLO26 semantic segmentation on Colab

**Before running:** get the project onto Colab using one of the options below, and put your dataset in Drive. That can be the `datasets/yolo_datasets/<region>/ultralytics/` folder built locally by `main.py build-dataset`, or a COCO export zip from `main.py build-dataset --export-format coco` or from Roboflow.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Open the project

Set `USE_GITHUB` below. Cloning the repo from GitHub is the simplest method to use this notebook.

In [ ]:
import os

USE_GITHUB = True

if USE_GITHUB:
    PROJECT_DIR = "/content/coastal-recession-ml"
    if not os.path.exists(PROJECT_DIR):
        !git clone https://github.com/lukasn9/coastal-recession-ml.git {PROJECT_DIR}
else:
    PROJECT_DIR = "/content/drive/MyDrive/coastal-recession-ml"

%cd {PROJECT_DIR}

In [ ]:
!pip install -q -r requirements.txt

## Restart the runtime

In case of any dependency issues, run the node below to restart the runtime.

In [ ]:
import os
os.kill(os.getpid(), 9)

## Choose your dataset source

Set `DATA_SOURCE` to `"local"` if you're pointing at an already-built `datasets/yolo_datasets/<region>/ultralytics/` folder (e.g. synced via Drive), or `"coco"` if you have a COCO-format zip (`coco_export.py` output, or a Roboflow export) to convert first.

In [ ]:
DATA_SOURCE = "local"  # "local" or "coco"

# --- if DATA_SOURCE == "local" ---
LOCAL_DATA_YAML = f"{PROJECT_DIR}/datasets/yolo_datasets/mediterranean/ultralytics/data.yaml"

# --- if DATA_SOURCE == "coco" ---
COCO_ZIP_PATH = "/content/drive/MyDrive/coastal-recession-ml/roboflow_export.zip"
COCO_EXTRACT_DIR = "/content/coco_export"
COCO_OUTPUT_DIR = "/content/drive/MyDrive/coastal-recession-ml/converted_coco_dataset"

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

if DATA_SOURCE == "coco":
    import shutil
    shutil.unpack_archive(COCO_ZIP_PATH, COCO_EXTRACT_DIR)

    from scripts.prepare_coco_dataset import find_split_dir, SPLIT_DIR_CANDIDATES
    from src.coco_to_masks import convert_coco_split
    from src.semantic_export import write_data_yaml
    from pathlib import Path

    input_dir = Path(COCO_EXTRACT_DIR)
    output_dir = Path(COCO_OUTPUT_DIR)
    train_dir = find_split_dir(input_dir, SPLIT_DIR_CANDIDATES["train"])
    val_dir = find_split_dir(input_dir, SPLIT_DIR_CANDIDATES["val"])

    class_names = None
    for split, split_dir in [("train", train_dir), ("val", val_dir)]:
        names = convert_coco_split(
            coco_json_path=split_dir / "_annotations.coco.json",
            images_dir=split_dir,
            output_images_dir=output_dir / "images" / split,
            output_masks_dir=output_dir / "masks" / split,
        )
        print(f"{split}: classes = {names}")
        class_names = class_names or names

    write_data_yaml(output_dir, class_names)
    DATA_YAML = str(output_dir / "data.yaml")
else:
    DATA_YAML = LOCAL_DATA_YAML

print("Training on:", DATA_YAML)

## Train

`project=` points into Drive, so weights (`last.pt` after every epoch, and `best.pt`) are saved on the cloud. If training disconnects partway through, resume by setting `model` to `.../weights/last.pt`.

In [ ]:
from src.training import train_model

results = train_model(
    data_yaml=DATA_YAML,
    model="yolo26n-sem.pt",
    epochs=100,
    imgsz=640,
    batch=16,
    device=None,  # auto-detects the Colab GPU
    seed=42,
    project="/content/drive/MyDrive/coastal-recession-ml/runs/train",
    name="coastal-seg",
)

## Next

Once training finishes, use `notebooks/predict_colab.ipynb` to run the trained weights on new images.